In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Long Decode and multi-segment continuation with Gemini 4

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Flong-context%2Fgemini_4_long_decode.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/long-context/gemini_4_long_decode.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/long-context/gemini_4_long_decode.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author(s) |
| --- |
| [Eric Dong](https://github.com/gericdong) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

Gemini 4 models (`gemini-4-neon`, `gemini-4-hydrogen`, `gemini-4-carbon`, and `gemini-4-argon`) share a unified **1M-token** window across input and output (output includes thinking tokens, function calls, and response text). When `max_output_tokens` is omitted, the default output budget is **64K** tokens. A single foreground segment generates up to **64K** output tokens on Neon and Hydrogen (streaming or non-streaming), or **64K** streaming / **32K** non-streaming on Carbon and Argon (with a 20-minute per-segment timeout).

Set `max_output_tokens` above 64K (up to 1M minus input tokens) to enable **Long Decode** across multiple segments on the `Interactions` API or `GenerateContent` (`v1beta1`; the OpenAI compatibility layer does not support Long Decode). See the [Gemini 4 Neon Long Decode guide](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/guides/gemini-4-neon#long-decodes) and [Get started with Gemini 4](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/start/get-started-with-gemini-4#long-decodes).

| Option | API surface | Server storage | Best for |
| :--- | :--- | :--- | :--- |
| **SDK automatic continuation (default)** | `Interactions` API & `GenerateContent` (`v1beta1`) | Optional (`store=True` / `False` on Interactions; none on `GenerateContent`) | Most streaming and non-streaming apps |
| **Background loop (`background=True`)** | `Interactions` API (`v1beta1`) | 7-day storage (`store=True` required) | Long async jobs without keeping a stream open |
| **Custom `continuation_token` loop** | `Interactions` API & `GenerateContent` (`v1beta1`) | Optional (`store=False` or `GenerateContent` for ZDR) | Custom retries and checkpoints |

Use Long Decode when one response needs more than 64K tokens of thinking and output (proofs, hard math, large multi-file code generation, or evals hitting thinking caps). Keep the default cap for chat, extraction, and short tool loops.

> **Note:** To keep this notebook under a minute to run, the custom continuation cells use a small initial `max_output_tokens` cap to trigger a real `continuation_token` and then raise the cap to finish. The same flow applies past 64K tokens.

## Get started

### Install Google Gen AI SDK

Automatic continuation requires `google-genai >= 2.34.0` for the Interactions API (`>= 2.30.1` for `GenerateContent`).

> **Note:** You can safely ignore `pip` dependency resolver errors from unrelated pre-installed packages in Colab.

In [ ]:
%pip install --upgrade --quiet google-genai

### Authenticate your notebook environment

Authenticate your account if running in Google Colab.

In [2]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [3]:
import os
import time
import uuid

from google import genai
from google.genai import errors
from google.genai import types

### Set Google Cloud project information

Set your Google Cloud project ID and initialize `genai.Client`. Long Decode requires the `v1beta1` API on Gemini agent (used by default unless you override `api_version="v1"`).

In [4]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "global"  # @param ["global", "us", "eu"]

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        timeout=240000,
        retry_options=types.HttpRetryOptions(attempts=3),
    ),
)

### Define model IDs

All Gemini 4 models support Long Decode up to 1M unified tokens. Track stored Interactions API IDs in `stored_ids` for cleanup at the end.

In [5]:
HYDROGEN_MODEL = "gemini-4-hydrogen"
NEON_MODEL = "gemini-4-neon"

MAX_CONTEXT_TOKENS = 1_048_576
RUN_TAG = uuid.uuid4().hex[:8]
stored_ids: list[str] = []

## Output budget guardrail

`max_output_tokens` caps total output (thinking, tool calls, and text) across all continuation segments. Passing a value above 1M returns `HTTP 400`. Use `thinking_level` to control reasoning depth; `max_output_tokens` only sets an upper cap.

In [6]:
try:
    client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents="Say hello.",
        config=types.GenerateContentConfig(
            max_output_tokens=MAX_CONTEXT_TOKENS + 1
        ),
    )
except errors.APIError as err:
    print(f"Expected guardrail -> HTTP {err.code}: {err.message[:125]}")

Expected guardrail -> HTTP 400: Unable to submit request because it has a maxOutputTokens value of 1048577 but the maximum supported value is 1048576. Update


## Automatic SDK continuation

`automatic_continuation` defaults to `True` on both `GenerateContent` and the `Interactions` API. Setting `max_output_tokens` above 64K lets the SDK follow `continuation_token` across segments until the model finishes or hits your cap.

### Stream with GenerateContent

In [7]:
# Setting max_output_tokens above 64K enables Long Decode.
# The SDK yields chunks across all continuation hops as one stream.
stream = client.models.generate_content_stream(
    model=NEON_MODEL,
    contents=(
        "List 4 design rules for a bounded work queue in Python covering "
        "backpressure, timeouts, and graceful shutdown."
    ),
    config=types.GenerateContentConfig(
        max_output_tokens=131_072,
        thinking_config=types.ThinkingConfig(thinking_level="MEDIUM"),
    ),
)

chunks: list[str] = []
final_usage = None
for chunk in stream:
    if chunk.text:
        chunks.append(chunk.text)
    if chunk.usage_metadata:
        final_usage = chunk.usage_metadata

print("".join(chunks)[:260] + "...\n")
print(
    f"prompt={final_usage.prompt_token_count}, "
    f"thoughts={final_usage.thoughts_token_count}, "
    f"output={final_usage.candidates_token_count}"
)

1. **Enforce a strict `maxsize` to apply backpressure:** Always initialize the queue with a finite bound (`queue.Queue(maxsize=N)` or `asyncio.Queue(maxsize=N)`). Decide explicitly how to handle a full queue: either block producers to slow down upstream ingest...

prompt=56, thoughts=855, output=340


### Automatic continuation on the Interactions API

Non-streaming `client.interactions.create()` merges all continuation segments into one `Interaction`, records every segment ID in `interaction.continuation_ids`, and preserves Hop 1's prompt size in `interaction.usage.initial_input_tokens`.

In [8]:
auto_turn = client.interactions.create(
    model=HYDROGEN_MODEL,
    input=(
        "List 4 failure modes in distributed consensus protocols and a 1-line "
        "mitigation for each."
    ),
    store=True,
    generation_config={
        "max_output_tokens": 131_072,
        "thinking_level": "medium",
    },
)
for seg_id in auto_turn.continuation_ids or [auto_turn.id]:
    stored_ids.append(seg_id)

print(f"Status: {auto_turn.status}")
print(f"Segments merged: {len(auto_turn.continuation_ids or [auto_turn.id])}")
print(
    f"initial_input_tokens={auto_turn.usage.initial_input_tokens}, "
    f"thought_tokens={auto_turn.usage.total_thought_tokens}, "
    f"output_tokens={auto_turn.usage.total_output_tokens}"
)

Status: completed
Segments merged: 1
initial_input_tokens=50, thought_tokens=289, output_tokens=204


## Server-managed background Long Decode

For jobs that run for several minutes without keeping a client stream open, set `background=True` and `store=True` on the Interactions API. The server runs the continuation loop and stores the merged result in one `Interaction`.

If you cancel a background job (`client.interactions.cancel(id=...)`) after 64K output tokens on Neon or Hydrogen (32K on Carbon or Argon), you can resume it from its checkpoint `continuation_token` (saved every 5K tokens thereafter). Stored chains can also switch between foreground (`background=False`) and `background=True` across segments.

In [9]:
bg_turn = client.interactions.create(
    model=HYDROGEN_MODEL,
    input=(
        "Write a concise 4-point production readiness checklist for running "
        "stateful LLM agents on Cloud Run."
    ),
    background=True,
    store=True,
    generation_config={
        "max_output_tokens": 256_000,
        "thinking_level": "low",
    },
)
stored_ids.append(bg_turn.id)
print(f"Initial status: {bg_turn.status}")

while bg_turn.status in ("queued", "in_progress"):
    time.sleep(2)
    bg_turn = client.interactions.get(id=bg_turn.id)
    print(f"Polled status:  {bg_turn.status}")

print(
    f"Done -> initial_input={bg_turn.usage.initial_input_tokens}, "
    f"thoughts={bg_turn.usage.total_thought_tokens}, "
    f"output={bg_turn.usage.total_output_tokens}"
)
print((bg_turn.output_text or "")[:300] + "...")

Initial status: in_progress
Polled status:  in_progress
Polled status:  in_progress
Polled status:  completed
Done -> initial_input=50, thoughts=302, output=323
Here is a 4-point production readiness checklist for running stateful LLM agents on Cloud Run:

1. **Externalize Agent State & Memory**
   * Cloud Run instances are ephemeral and can scale to zero. Persist conversation history, intermediate reasoning steps, and user context in external managed stora...


## Stored continuation loop and budget expansion

In a custom loop (`automatic_continuation=False`), check two continuation states on the Interactions API:
- `status == "requires_continuation"`: hit the per-segment cap before `max_output_tokens`. Pass `previous_interaction_id` and `continuation_token` (without `input`) to continue.
- `status == "incomplete"`: hit your `max_output_tokens` cap (including thinking-only segments where `output_text` is empty). Continuing without raising `max_output_tokens` returns `HTTP 400`; raising it resumes decoding.

Rules enforced across continuation segments:
- Starting a new user turn (`previous_interaction_id` with new `input` and no `continuation_token`) from an `incomplete` or `requires_continuation` segment returns `HTTP 400`.
- Keep `model`, `system_instruction`, `tools`, and `thinking_level` unchanged across continuation segments (changing `model`, `system_instruction`, or function declarations returns `HTTP 400`).

Here, Segment 1 sets `max_output_tokens=600` to trigger `incomplete` right away, verifies the `HTTP 400` error, and then raises the budget to finish.

In [10]:
code_prompt = (
    "Write a concise Go rate-limiter package with 3 numbered sections: "
    "1. TokenBucket struct, 2. Allow() method, 3. Unit test."
)

# Segment 1: tight budget forces status="incomplete" and emits continuation_token
seg1 = client.interactions.create(
    model=HYDROGEN_MODEL,
    input=code_prompt,
    store=True,
    generation_config={"max_output_tokens": 600, "thinking_level": "low"},
    automatic_continuation=False,
)
stored_ids.append(seg1.id)
print(
    f"Segment 1 -> status={seg1.status}, has_token={bool(seg1.continuation_token)}, "
    f"text_chars={len(seg1.output_text or '')}, "
    f"initial_in={seg1.usage.initial_input_tokens}, "
    f"total_in={seg1.usage.total_input_tokens}"
)

# Guardrail: resuming without raising max_output_tokens returns HTTP 400
try:
    client.interactions.create(
        model=HYDROGEN_MODEL,
        previous_interaction_id=seg1.id,
        continuation_token=seg1.continuation_token,
        generation_config={"max_output_tokens": 600, "thinking_level": "low"},
        automatic_continuation=False,
    )
except Exception as err:
    print(f"Expected guardrail -> {type(err).__name__}: {str(err)[:165]}")

# Segment 2: raise max_output_tokens and omit input (stored in previous_interaction_id)
seg2 = client.interactions.create(
    model=HYDROGEN_MODEL,
    previous_interaction_id=seg1.id,
    continuation_token=seg1.continuation_token,
    generation_config={"max_output_tokens": 8192, "thinking_level": "low"},
    automatic_continuation=False,
)
stored_ids.append(seg2.id)
print(
    f"Segment 2 -> status={seg2.status}, has_token={bool(seg2.continuation_token)}, "
    f"text_chars={len(seg2.output_text or '')}, "
    f"initial_in={seg2.usage.initial_input_tokens}, "
    f"total_in={seg2.usage.total_input_tokens}"
)

Segment 1 -> status=incomplete, has_token=True, text_chars=900, initial_in=65, total_in=65
Expected guardrail -> BadRequestError: Error code: 400 - {'error': {'message': 'max_output_tokens (600) has already been exhausted. Because max_output_tokens is accumulated across continuation requests, p
Segment 2 -> status=completed, has_token=False, text_chars=344, initial_in=65, total_in=665


## Stateless continuation and implicit cache reuse

For Zero Data Retention (ZDR) workflows, avoid server storage by using either `store=False` on the Interactions API or `GenerateContent` (`v1beta1`). Both pass intermediate state encrypted inside `continuation_token`.

### Stateless Interactions API loop (`store=False`)

With `store=False`, omit `previous_interaction_id` and resend the original `input` alongside `continuation_token`. You can switch from `store=True` to `store=False` mid-decode by resending `input`, but switching from `store=False` to `store=True` is not supported.

In [11]:
stateless_prompt = (
    "Write a short Python thread-safe LRU cache class with get, put, and one unit test."
)

stateless_1 = client.interactions.create(
    model=HYDROGEN_MODEL,
    input=stateless_prompt,
    store=False,
    generation_config={"max_output_tokens": 500, "thinking_level": "low"},
    automatic_continuation=False,
)
print(
    f"Stateless Hop 1 -> status={stateless_1.status}, "
    f"has_token={bool(stateless_1.continuation_token)}, "
    f"text_chars={len(stateless_1.output_text or '')}"
)

if stateless_1.continuation_token:
    stateless_2 = client.interactions.create(
        model=HYDROGEN_MODEL,
        input=stateless_prompt,
        store=False,
        continuation_token=stateless_1.continuation_token,
        generation_config={"max_output_tokens": 4096, "thinking_level": "low"},
        automatic_continuation=False,
    )
    print(
        f"Stateless Hop 2 -> status={stateless_2.status}, "
        f"text_chars={len(stateless_2.output_text or '')}, "
        f"initial_in={stateless_2.usage.initial_input_tokens}, "
        f"total_in={stateless_2.usage.total_input_tokens}"
    )

Stateless Hop 1 -> status=incomplete, has_token=True, text_chars=0
Stateless Hop 2 -> status=completed, text_chars=1281, initial_in=51, total_in=551


### GenerateContent continuation hops and implicit caching

In a custom `GenerateContent` loop, do not append partial output to `contents` or ask the model to continue. Resend the original `contents`, `system_instruction`, and `tools` with `continuation_token`.

- Loop while `finish_reason` is `CONTINUATION` (single-segment cap reached) and `continuation_token` is present, even on thinking-only hops (`text == ""`).
- If `finish_reason` is `MAX_TOKENS` and `continuation_token` is present (shown below with small budgets), raise `max_output_tokens` on the next hop to resume.
- Merge accumulated parts into one `model` turn for multi-turn history only after `STOP`.

Each hop sends the prompt plus prior decoded tokens as input, so `prompt_token_count` grows across hops and hits **95% Implicit Context Caching** once the shared prefix reaches 4K tokens. Each hop picks up mid-token where the previous hop stopped.

In [12]:
# Static >4K-token system_instruction so later hops qualify for implicit caching
static_specs = f"Service catalog (run {RUN_TAG}):\n" + "\n".join(
    f"- svc-{i:03d}: protocol=gRPC, timeout_ms={100 + i}, max_retries=3, "
    "idempotency=required, owner=platform-core"
    for i in range(1, 320)
)
gc_prompt = (
    "Using svc-001 settings, write a complete Go token-bucket limiter with "
    "Allow() and a unit test."
)

cont_token = None
hop_texts: list[str] = []
for hop_idx, budget in enumerate((500, 1000, 4096), start=1):
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=gc_prompt,
        config=types.GenerateContentConfig(
            system_instruction=static_specs,
            max_output_tokens=budget,
            continuation_token=cont_token,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
            automatic_continuation=False,
        ),
    )
    cand = resp.candidates[0]
    u = resp.usage_metadata
    cached = u.cached_content_token_count or 0
    hit_pct = 100.0 * cached / u.prompt_token_count
    hop_texts.append(resp.text or "")
    print(
        f"Hop {hop_idx} -> reason={cand.finish_reason.name:<10} | "
        f"prompt={u.prompt_token_count:>5} | cached={cached:>5} ({hit_pct:4.1f}%) | "
        f"thoughts={u.thoughts_token_count or 0:>3} | out={u.candidates_token_count or 0:>4}"
    )
    cont_token = cand.continuation_token
    if not cont_token:
        break
    time.sleep(2.0)

if len(hop_texts) >= 2:
    print(f"Seam  -> Hop 1 tail: {repr(hop_texts[0][-45:])}")
    print(f"Seam  -> Hop 2 head: {repr(hop_texts[1][:45])}")

Hop 1 -> reason=MAX_TOKENS | prompt=12194 | cached=    0 ( 0.0%) | thoughts=412 | out=  88
Hop 2 -> reason=MAX_TOKENS | prompt=12694 | cached=10140 (79.9%) | thoughts=  0 | out= 500
Hop 3 -> reason=STOP       | prompt=13194 | cached=10645 (80.7%) | thoughts=  0 | out= 810
Seam  -> Hop 1 tail: ' + `max_retries=3`)\n- **Refill interval:** `1'
Seam  -> Hop 2 head: '01ms` per token (derived from `timeout_ms=101'


## Context compaction and interruption recovery

- Base context compaction checks on `usage.initial_input_tokens` (Interactions API) or Hop 1's `prompt_token_count` (`GenerateContent`), because `total_input_tokens` and `prompt_token_count` grow as prior decoded tokens are sent as input across hops. Wait until no `continuation_token` is active before compacting.
- If a segment fails, times out, or disconnects during `interactions.create(..., automatic_continuation=True)` after a hop completes or a resumable checkpoint is reached, the SDK raises `errors.ContinuationInterruptedError` with `err.continuation_token`, `err.interaction_id`, `err.partial_interaction` (non-streaming), and `err.continuation_ids` so you can resume from the checkpoint.

In [13]:
def should_compact_history(
    initial_input_tokens: int,
    status: str,
    continuation_token: str | bytes | None,
    threshold_tokens: int = 600_000,
) -> tuple[bool, str]:
    if continuation_token or status in ("requires_continuation", "incomplete"):
        return False, "Wait: decode is still mid-continuation."
    if initial_input_tokens >= threshold_tokens:
        return True, f"Compact: initial input ({initial_input_tokens:,}) >= threshold."
    return False, f"Keep: initial input ({initial_input_tokens:,}) is below threshold."


for label, args in [
    ("Mid-decode hop (incomplete)", (seg1.usage.initial_input_tokens, seg1.status, seg1.continuation_token)),
    ("Completed decode (small input)", (seg2.usage.initial_input_tokens, seg2.status, seg2.continuation_token)),
    ("Completed decode (650K input)", (650_000, "completed", None)),
]:
    compact, reason = should_compact_history(*args)
    print(f"{label:<32} -> compact={str(compact):<5} | {reason}")

Mid-decode hop (incomplete)      -> compact=False | Wait: decode is still mid-continuation.
Completed decode (small input)   -> compact=False | Keep: initial input (65) is below threshold.
Completed decode (650K input)    -> compact=True  | Compact: initial input (650,000) >= threshold.


## Cleaning up

Delete stored Interactions API segments created in this notebook in reverse order (last segment first; stored interactions otherwise expire after 7 days).

In [14]:
deleted = 0
for interaction_id in reversed(stored_ids):
    client.interactions.delete(id=interaction_id)
    deleted += 1

print(f"Deleted {deleted} stored interactions.")

Deleted 4 stored interactions.
